# DSPy Adapters

This notebook demonstrates how to use DSPy 3.x adapters to control prompt formatting and output structure.

**What You'll Learn:**
- How to use `ChatAdapter` for conversational interfaces
- How to use `JSONAdapter` for structured JSON outputs
- How to use `XMLAdapter` for XML-structured data
- When to use each adapter type
- How to combine adapters with modules

## Setup

In [ ]:
import os
import sys
import json
sys.path.append('../../')

import dspy
from utils import (
    setup_default_lm, configure_dspy,
    create_chat_adapter, create_json_adapter, create_xml_adapter,
    print_step, print_result, print_error
)
from dotenv import load_dotenv

load_dotenv('../../.env')

## Part 1: ChatAdapter

The `ChatAdapter` formats messages in chat format, which is more natural for conversational AI applications.

In [ ]:
try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    chat_adapter = create_chat_adapter()
    configure_dspy(lm=lm, adapter=chat_adapter)
    print_result("Language model configured with ChatAdapter!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")

In [ ]:
# Example 1.1: Basic Conversational Interface
print_step("Basic Chat", "Simple conversational interface")

class ChatAssistant(dspy.Signature):
    """A friendly conversational assistant."""
    user_message = dspy.InputField(desc="The user's message")
    assistant_response = dspy.OutputField(desc="A helpful and friendly response")

chat_bot = dspy.Predict(ChatAssistant)

messages = [
    "Hi! How are you?",
    "Can you help me understand what DSPy is?",
    "That's interesting! What are adapters used for?"
]

for msg in messages:
    result = chat_bot(user_message=msg)
    print(f"\nUser: {msg}")
    print(f"Assistant: {result.assistant_response}")

In [ ]:
# Example 1.2: Multi-turn Conversation with Context
print_step("Multi-turn Chat", "Contextual conversation")

class ContextualChat(dspy.Module):
    """A chatbot that maintains conversation context."""
    def __init__(self):
        super().__init__()

        class GenerateResponse(dspy.Signature):
            """Generate a contextual response."""
            conversation_history = dspy.InputField(desc="Previous conversation")
            user_message = dspy.InputField(desc="Current user message")
            response = dspy.OutputField(desc="Contextual response")

        self.respond = dspy.Predict(GenerateResponse)

    def forward(self, history, message):
        history_text = "\n".join([f"{h['role']}: {h['content']}" for h in history])
        result = self.respond(conversation_history=history_text, user_message=message)
        return dspy.Prediction(response=result.response)

contextual_bot = ContextualChat()
conversation = []
test_messages = [
    "What's the capital of France?",
    "What's the population of that city?",
    "What are some famous landmarks there?"
]

for msg in test_messages:
    result = contextual_bot(history=conversation, message=msg)
    print(f"\nUser: {msg}")
    print(f"Bot: {result.response}")
    conversation.append({"role": "user", "content": msg})
    conversation.append({"role": "assistant", "content": result.response})

## Part 2: JSONAdapter

The `JSONAdapter` encourages the model to produce valid JSON outputs, making it easier to parse structured data.

In [ ]:
# Reconfigure with JSONAdapter
json_adapter = create_json_adapter()
lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=1000)
configure_dspy(lm=lm, adapter=json_adapter)
print_result("Reconfigured with JSONAdapter", "Status")

In [ ]:
# Example 2.1: Structured Data Extraction
print_step("JSON Data Extraction", "Extract structured data as JSON")

class ExtractPersonInfo(dspy.Signature):
    """Extract person information in JSON format."""
    text = dspy.InputField(desc="Text containing person information")
    person_json = dspy.OutputField(desc="JSON with fields: name, age, occupation, location")

extractor = dspy.Predict(ExtractPersonInfo)

sample_text = """
John Smith is a 35-year-old software engineer living in San Francisco.
He has been working in the tech industry for over 10 years.
"""

result = extractor(text=sample_text)
print(f"Input text: {sample_text.strip()}")
print(f"\nExtracted JSON:")
try:
    parsed = json.loads(result.person_json)
    print(json.dumps(parsed, indent=2))
    print("\nValid JSON extracted!")
except json.JSONDecodeError:
    print(result.person_json)
    print("\nOutput may not be valid JSON (adapter helps but doesn't guarantee)")

In [ ]:
# Example 2.2: Complex JSON Structure
print_step("Complex JSON", "Generate product catalog as JSON")

class GenerateProductCatalog(dspy.Signature):
    """Generate a product catalog in JSON format."""
    category = dspy.InputField(desc="Product category")
    num_products = dspy.InputField(desc="Number of products to generate")
    catalog_json = dspy.OutputField(
        desc="JSON array of products, each with: id, name, price, description, features (array)"
    )

catalog_gen = dspy.Predict(GenerateProductCatalog)

result = catalog_gen(category="Laptops", num_products="3")
print("Generated catalog JSON:")
try:
    catalog = json.loads(result.catalog_json)
    print(json.dumps(catalog, indent=2))
    print(f"\nGenerated {len(catalog)} products")
except json.JSONDecodeError:
    print(result.catalog_json)

In [ ]:
# Example 2.3: JSON Validation and Correction
print_step("JSON Validation", "Validate and fix invalid JSON")

class ValidateAndFixJSON(dspy.Module):
    """Validate JSON against a schema and fix if needed."""
    def __init__(self):
        super().__init__()

        class FixJSON(dspy.Signature):
            """Fix invalid JSON to match schema."""
            invalid_json = dspy.InputField(desc="Potentially invalid JSON")
            schema_description = dspy.InputField(desc="Description of required schema")
            fixed_json = dspy.OutputField(desc="Valid JSON matching the schema")

        self.fixer = dspy.Predict(FixJSON)

    def forward(self, data, schema):
        try:
            parsed = json.loads(data)
            return dspy.Prediction(is_valid=True, data=data, message="JSON is valid")
        except json.JSONDecodeError:
            result = self.fixer(invalid_json=data, schema_description=schema)
            return dspy.Prediction(is_valid=False, data=result.fixed_json, message="JSON was fixed")

validator = ValidateAndFixJSON()

invalid_json = '{"name": "John", "age": 30, occupation: "Engineer"}'
schema = "Object with name (string), age (number), occupation (string)"

result = validator(data=invalid_json, schema=schema)
print(f"Original: {invalid_json}")
print(f"Fixed: {result.data}")
print(f"Status: {result.message}")

## Part 3: XMLAdapter

The `XMLAdapter` formats outputs as XML, useful for legacy systems and hierarchical data.

In [ ]:
# Reconfigure with XMLAdapter
xml_adapter = create_xml_adapter()
lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=1000)
configure_dspy(lm=lm, adapter=xml_adapter)
print_result("Reconfigured with XMLAdapter", "Status")

In [ ]:
# Example 3.1: Generate XML Configuration
print_step("XML Config", "Generate XML configuration")

class GenerateXMLConfig(dspy.Signature):
    """Generate XML configuration."""
    app_name = dspy.InputField(desc="Application name")
    settings = dspy.InputField(desc="Configuration settings to include")
    xml_config = dspy.OutputField(desc="XML configuration document")

xml_gen = dspy.Predict(GenerateXMLConfig)

result = xml_gen(
    app_name="MyApp",
    settings="database connection, API endpoints, logging level"
)
print("Generated XML:")
print(result.xml_config)

In [ ]:
# Example 3.2: Convert Structured Data to XML
print_step("Data to XML", "Convert structured data to XML format")

class DataToXML(dspy.Signature):
    """Convert structured data to XML format."""
    data_description = dspy.InputField(desc="Description of data")
    xml_output = dspy.OutputField(desc="XML representation of the data")

converter = dspy.Predict(DataToXML)

data_desc = """
A company with name 'TechCorp', employees: [
    {name: 'Alice', role: 'Engineer', salary: 100000},
    {name: 'Bob', role: 'Manager', salary: 120000}
]
"""

result = converter(data_description=data_desc)
print(f"Data: {data_desc.strip()}")
print(f"\nXML output:")
print(result.xml_output)

## Part 4: Adapter Comparison

Understanding when to use each adapter.

In [ ]:
print("ChatAdapter:")
print("  Use for: Conversational interfaces, chatbots, dialogue systems")
print("  Benefits: More natural conversation flow, better context handling")
print("  Best with: Chat-optimized models (GPT-4, Claude)")
print()
print("JSONAdapter:")
print("  Use for: APIs, structured data extraction, configuration generation")
print("  Benefits: Encourages valid JSON outputs, easier parsing")
print("  Best with: Any model, especially for data-heavy applications")
print()
print("XMLAdapter:")
print("  Use for: Legacy systems, document generation, configuration files")
print("  Benefits: Hierarchical structure, good for complex nested data")
print("  Best with: Any model, especially when XML is required")
print()
print("No Adapter (Default):")
print("  Use for: General text generation, simple Q&A, open-ended tasks")
print("  Benefits: Maximum flexibility, no format constraints")

## Part 5: Advanced Patterns

Combining adapters with complex workflows for multi-format processing.

In [ ]:
class MultiFormatProcessor(dspy.Module):
    """Process data and output in multiple formats using different adapters."""
    def __init__(self):
        super().__init__()

        class ProcessData(dspy.Signature):
            """Process and format data."""
            input_data = dspy.InputField()
            format_type = dspy.InputField(desc="json or xml")
            output = dspy.OutputField(desc="Formatted output")

        self.processor = dspy.Predict(ProcessData)

    def forward(self, data, format="json"):
        result = self.processor(input_data=data, format_type=format)
        return dspy.Prediction(output=result.output, format=format)

processor = MultiFormatProcessor()

print("This pattern demonstrates:")
print("  - Dynamic adapter selection based on requirements")
print("  - Multi-format output generation")
print("  - Flexible data processing pipelines")
print("\nUse cases:")
print("  - API services that support multiple output formats")
print("  - Data transformation pipelines")
print("  - Format conversion tools")

## Summary

**Key Takeaways:**
1. Adapters control how prompts are formatted and parsed
2. `ChatAdapter`: Best for conversational interfaces
3. `JSONAdapter`: Best for structured data and APIs
4. `XMLAdapter`: Best for hierarchical data and legacy systems
5. Use `configure_dspy(adapter=...)` to set the adapter
6. Adapters improve output reliability but don't guarantee format
7. Choose adapters based on your application's needs